In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os


In [ ]:
try:
    spark.stop()
except:
    pass


In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_FactClaimHeader")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())


In [ ]:
silver_claim = "../../data_lake/silver/silver_claim_header/"
silver_eob = "../../data_lake/silver/silver_eob_header/"
gold_base_path = "../../data_lake/gold/fact_claim_header/"
gold_dimpatient = "../../data_lake/gold/dim_patient/"
gold_dimpractitioner = "../../data_lake/gold/dim_practitioner/"
gold_dimorganization = "../../data_lake/gold/dim_organization/"
gold_dimlocation = "../../data_lake/gold/dim_location/"
gold_dimdate = "../../data_lake/gold/dim_date/"


In [ ]:
df_claim = spark.read.format("parquet").load(silver_claim)
df_eob = spark.read.format("parquet").load(silver_eob)
df_dimpatient = spark.read.format("parquet").load(gold_dimpatient)
df_dimpractitioner = spark.read.format("parquet").load(gold_dimpractitioner)
df_dimorganization = spark.read.format("parquet").load(gold_dimorganization)
df_dimlocation = spark.read.format("parquet").load(gold_dimlocation)
df_dimdate = spark.read.format("parquet").load(gold_dimdate)


In [ ]:
df_claim_eob = (df_claim.alias("clm")
    .join(df_eob.alias("eob"), col("clm.claim_id") == col("eob.claim_id"), "left"))


In [ ]:
df_inter = (df_claim_eob
    .join(df_dimpatient.alias("pat"), col("clm.patient_id") == col("pat.patient_id"), "left")
    .join(df_dimpractitioner.alias("prac"), col("eob.practitioner_npi") == col("prac.npi"), "left")
    .join(df_dimorganization.alias("org"), col("clm.provider_id") == col("org.organization_id"), "left")
    .join(df_dimlocation.alias("loc"), col("clm.facility_id") == col("loc.location_id"), "left")
    .join(df_dimdate.alias("d_created"), col("clm.created").cast("date") == col("d_created.date"), "left")
    .join(df_dimdate.alias("d_start"), col("clm.billable_period_start").cast("date") == col("d_start.date"), "left")
    .join(df_dimdate.alias("d_end"), col("clm.billable_period_end").cast("date") == col("d_end.date"), "left")
    .select(
        F.conv(F.substring(F.md5(col("clm.claim_id")), 1, 15), 16, 10).cast("bigint").alias("claim_key"),
        col("clm.claim_id"),
        col("pat.patient_key"),
        col("prac.practitioner_key"),
        col("org.organization_key").alias("provider_organization_key"),
        col("loc.location_key").alias("facility_key"),
        col("d_created.date_key").alias("created_date_key"),
        col("d_start.date_key").alias("service_start_date_key"),
        col("d_end.date_key").alias("service_end_date_key"),
        col("clm.status").alias("claim_status"),
        col("clm.claim_type"),
        col("clm.use").alias("claim_use"),
        col("eob.payer_name"),
        col("eob.total_submitted_amt").cast("double").alias("total_submitted_amt"),
        col("eob.payment_amt").cast("double").alias("payment_amt"),
        col("eob.outcome"),
        F.current_timestamp().alias("gold_timestamp")
    )
)


In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)


In [ ]:
spark.stop()
